# TFT Hyperparameter Sweep — Kaggle GPU T4 (two-stage, resumable)
v1 TFT (hidden 32, lr 1e-3, dropout 0.15, heads 4 — ~86K params) was hand-picked, never swept.
This notebook answers "was that the best architecture?" honestly:

- **Stage A (screening):** 6 configs x 6 epochs each, ranked by **val green-flag MAE**. ~2.5-3.5 h.
- **Stage B (convergence):** top-2 configs trained to convergence (max 40 epochs, EarlyStopping
  patience 6 — kills the v2 undertraining confound). ~3-5 h.
- **Winner guard:** only if the best converged config beats the deployed v1 bar
  (val green **1.12 s**) does it become `models/tft_lap.{ckpt,pt}` + get recalibrated.
  Otherwise v1 stays and the sweep result is "v1 confirmed near-optimal" — also reportable.
- **Test set is touched ONLY for the final winner** (selection is val-only, project rule).

**Before running:**
1. Notebook settings: **Accelerator = GPU T4 x2 or T4** (NOT P100 — Error 15), **Internet = ON**,
   **Persistence = Files only** (REQUIRED for resume — stage CSVs + checkpoints live in /kaggle/working).
2. **+ Add Input** -> attach the `tft_full_data` dataset (the `laps_*_r*.parquet` files, same as runs 04/05).
3. Repo pushed to GitHub `main` (cell 2 clones/pulls it).

**Total wall time ~6-8 h.** If the session dies, just re-run all cells: completed trials are
skipped via `hpo_stage_*.csv` (resume is per-trial, not per-epoch).

**Known caveat (documented):** 6-epoch screening favours fast-converging learning rates;
mitigated by converging the top-2 (not top-1) in Stage B.

**When done:** download `tft_hpo_artifacts.zip` from the Output panel. Local steps in the last cell.

In [ ]:
# 1. Pinned triangle on top of Kaggle's stock cu128 torch. T4 only (Error 15).
!pip -q install 'pytorch-forecasting==1.7.0' 'lightning==2.6.5' mlflow fastf1 pandera scipy
import torch, pytorch_forecasting as pf, lightning
print('pf', pf.__version__, '| lightning', lightning.__version__, '| torch', torch.__version__)
print('cuda:', torch.cuda.is_available(), '| device:', torch.cuda.get_device_name(0))
assert torch.cuda.get_device_capability(0) in {(7,5),(8,0),(8,6),(9,0)}, \
    'Switch Accelerator to T4 — this GPU arch is not in the torch build.'

In [ ]:
# 2. Clone repo (or pull) + path
import os, sys
REPO = '/kaggle/working/f1-strategist'
if not os.path.exists(REPO):
    !git clone https://github.com/Shreyansh262/f1-strategist.git $REPO
else:
    !cd $REPO && git pull
sys.path.insert(0, REPO)
os.chdir(REPO)
!cd $REPO && git log --oneline -1

In [ ]:
# 3. Copy parquets from the attached dataset into data/raw
import glob, shutil, pathlib
dst = pathlib.Path(REPO) / 'data' / 'raw'
dst.mkdir(parents=True, exist_ok=True)
src_files = glob.glob('/kaggle/input/**/laps_*_r*.parquet', recursive=True)
assert src_files, 'No laps_*_r*.parquet under /kaggle/input/ — attach the data zip as a Dataset first.'
for f in src_files:
    shutil.copy(f, dst / pathlib.Path(f).name)
copied = sorted(p.name for p in dst.glob('laps_*_r*.parquet'))
print(len(copied), 'files | seasons:', sorted({n.split('_')[1] for n in copied}))

In [ ]:
# 4. Load data ONCE, build datasets with the V1 feature roles (v1 is the champion the
#    sweep must beat on equal features; Driver/EngineMaker is the separate v2 tiebreak, cell 12).
import pandas as pd, torch
from pathlib import Path
from src.models.lap_time.train_tft import (
    make_datasets, prepare, green_mae, breakdown, quantile_coverage, export_cpu,
    QUANTILES, STATIC_CATEGORICALS_V1, STATIC_CATEGORICALS, VAL_SEASONS, TEST_SEASONS)
from src.models.lap_time.train_tft_data import load_tft_data

df = load_tft_data()   # carry-back loader: Team/Compound/StintID/TrackStatus re-attached
training, validation, test = make_datasets(df, static_categoricals=STATIC_CATEGORICALS_V1)
BS = 512
train_dl = training.to_dataloader(train=True,  batch_size=BS,   num_workers=4, pin_memory=True)
val_dl   = validation.to_dataloader(train=False, batch_size=BS*2, num_workers=4)
test_dl  = test.to_dataloader(train=False,       batch_size=BS*2, num_workers=4)
VAL_RAW  = prepare(df[df['Season'].isin(VAL_SEASONS)])
TEST_RAW = prepare(df[df['Season'].isin(TEST_SEASONS)])
REPORTS  = Path(REPO) / 'reports' / 'lap_time'; REPORTS.mkdir(parents=True, exist_ok=True)
print(f'{len(df)} laps | train windows {len(training)} | val {len(validation)} | test {len(test)}')

In [ ]:
# 5. Trial runner + per-trial resume. Selection metric = val GREEN-FLAG MAE (the project
#    headline metric, same as model_comparison.csv) — NOT raw val_loss.
import gc, time, shutil
import lightning.pytorch as pl
from lightning.pytorch.callbacks import EarlyStopping, ModelCheckpoint
from pytorch_forecasting import TemporalFusionTransformer
from pytorch_forecasting.metrics import QuantileLoss

HPO_DIR = Path('/kaggle/working/hpo'); HPO_DIR.mkdir(exist_ok=True)

def run_trial(name, cfg, max_epochs, patience=None, train_ds=None, tdl=None, vdl=None, raw=None):
    pl.seed_everything(42, workers=True)   # same seed every trial -> configs comparable
    tft = TemporalFusionTransformer.from_dataset(
        train_ds,
        learning_rate=cfg['lr'],
        hidden_size=cfg['hidden'],
        attention_head_size=cfg['heads'],
        dropout=cfg['dropout'],
        hidden_continuous_size=cfg['hidden'] // 2,
        loss=QuantileLoss(quantiles=QUANTILES),
        log_interval=-1,
        optimizer='adamw',
        reduce_on_plateau_patience=4,
    )
    n_params = sum(p.numel() for p in tft.parameters())
    ckpt = ModelCheckpoint(dirpath=str(HPO_DIR / name), filename='best',
                           monitor='val_loss', mode='min', save_top_k=1)
    callbacks = [ckpt] + ([EarlyStopping(monitor='val_loss', patience=patience, mode='min')]
                          if patience else [])
    trainer = pl.Trainer(max_epochs=max_epochs,
                         accelerator='gpu' if torch.cuda.is_available() else 'cpu', devices=1,
                         gradient_clip_val=0.1, callbacks=callbacks,
                         logger=False, enable_progress_bar=False, enable_model_summary=False)
    t0 = time.time()
    trainer.fit(tft, tdl, vdl)
    best = TemporalFusionTransformer.load_from_checkpoint(ckpt.best_model_path)
    mae = green_mae(best, vdl, raw)        # val only — test stays untouched until the winner
    row = dict(trial=name, **cfg, n_params=n_params, epochs=trainer.current_epoch,
               minutes=round((time.time() - t0) / 60, 1),
               val_mae_all=round(mae['mae_all'], 4), val_mae_green=round(mae['mae_green'], 4),
               ckpt=ckpt.best_model_path)
    del tft, best, trainer; gc.collect(); torch.cuda.empty_cache()
    return row

def run_stage(configs, csv_path, **kw):
    done = pd.read_csv(csv_path) if csv_path.exists() else pd.DataFrame()
    for name, cfg in configs:
        if not done.empty and name in set(done['trial']):
            print('skip (already done):', name); continue
        print(f'>>> {name} {cfg}', flush=True)
        row = run_trial(name, cfg, **kw)
        done = pd.concat([done, pd.DataFrame([row])], ignore_index=True)
        done.to_csv(csv_path, index=False)   # checkpoint the stage after EVERY trial
        print(row, flush=True)
    return done

## Stage A — screening (6 configs x 6 epochs, ~2.5-3.5 h)
Axes: hidden_size {16, 32, 64} x learning_rate {3e-4, 1e-3, 3e-3}, dropout/heads loosely tied
(fractional design, not full 36-cell grid — budget). `a1_v1anchor` IS the v1 config, so the
ranking has a known reference point at equal epochs.

In [ ]:
# 6. Stage A
STAGE_A = [
    ('a1_v1anchor',  dict(hidden=32, lr=1e-3, dropout=0.15, heads=4)),
    ('a2_small',     dict(hidden=16, lr=1e-3, dropout=0.10, heads=2)),
    ('a3_big',       dict(hidden=64, lr=1e-3, dropout=0.20, heads=4)),
    ('a4_lowlr',     dict(hidden=32, lr=3e-4, dropout=0.15, heads=4)),
    ('a5_highlr',    dict(hidden=32, lr=3e-3, dropout=0.15, heads=4)),
    ('a6_big_lowlr', dict(hidden=64, lr=3e-4, dropout=0.10, heads=4)),
]
SCREEN_EPOCHS = 6
CSV_A = REPORTS / 'hpo_stage_a.csv'
stage_a = run_stage(STAGE_A, CSV_A, max_epochs=SCREEN_EPOCHS,
                    train_ds=training, tdl=train_dl, vdl=val_dl, raw=VAL_RAW)
print(stage_a.sort_values('val_mae_green').to_string(index=False))

## Stage B — converge the top-2 (max 40 epochs, EarlyStopping patience 6, ~3-5 h)
Top-2 (not top-1) hedges the short-screening bias toward fast learning rates.

In [ ]:
# 7. Stage B
TOP_K = 2   # set to 1 if short on session time
stage_a = pd.read_csv(CSV_A).sort_values('val_mae_green')
top = stage_a.head(TOP_K)
STAGE_B = [(f'b_{r.trial}', dict(hidden=int(r.hidden), lr=float(r.lr),
                                 dropout=float(r.dropout), heads=int(r.heads)))
           for r in top.itertuples()]
print('Stage B configs:', STAGE_B)
CSV_B = REPORTS / 'hpo_stage_b.csv'
stage_b = run_stage(STAGE_B, CSV_B, max_epochs=40, patience=6,
                    train_ds=training, tdl=train_dl, vdl=val_dl, raw=VAL_RAW)
print(stage_b.sort_values('val_mae_green').to_string(index=False))

In [ ]:
# 8. Winner guard: deploy ONLY if the converged best beats v1's val green bar.
#    Test set evaluated here for the first time (winner only).
V1_VAL_GREEN = 1.12   # deployed v1 TFT, val green MAE (model_comparison.csv)
stage_b = pd.read_csv(CSV_B).sort_values('val_mae_green')
win = stage_b.iloc[0]
print(stage_b.to_string(index=False), '\n')

if win.val_mae_green < V1_VAL_GREEN:
    print(f'WINNER {win.trial}: val green {win.val_mae_green:.3f} BEATS v1 {V1_VAL_GREEN} -> deploying')
    best = TemporalFusionTransformer.load_from_checkpoint(win.ckpt)
    bd_frames, cov_rows = [], []
    for split, dl, raw in [('val', val_dl, VAL_RAW), ('test', test_dl, TEST_RAW)]:
        bd = breakdown(best, dl, raw); bd.insert(0, 'split', split); bd_frames.append(bd)
        cov_rows.append({'split': split, **quantile_coverage(best, dl, raw)})
    pd.concat(bd_frames, ignore_index=True).to_csv(REPORTS / 'tft_breakdown.csv', index=False)
    pd.DataFrame(cov_rows).to_csv(REPORTS / 'tft_calibration.csv', index=False)
    MODELS = Path(REPO) / 'models'; MODELS.mkdir(exist_ok=True)
    shutil.copy(win.ckpt, MODELS / 'tft_lap.ckpt')
    export_cpu(best, MODELS / 'tft_lap.pt')
    # conformal recalibration on the FRESH checkpoint (green-only, era-aware)
    from src.models.lap_time.recalibrate import main as recal
    recal()
else:
    print(f'No config beat v1 ({V1_VAL_GREEN}) on val green. KEEP v1.')
    print('Result is still reportable: "v1 confirmed near-optimal under a 2-stage sweep".')
    print('Do NOT overwrite the local models/tft_lap.* with anything from this run.')

## Optional — v2 feature tiebreak (flip the flag; ~2-2.5 h extra)
The clean answer to "did Driver+EngineMaker fail on merit or undertraining?": best swept
architecture + v2 static categoricals, trained to convergence under the same early-stop rules.

In [ ]:
# 9. OPTIONAL converged v2-features run (off by default — session time)
RUN_V2_TIEBREAK = False
if RUN_V2_TIEBREAK:
    training2, validation2, _ = make_datasets(df, static_categoricals=STATIC_CATEGORICALS)
    tdl2 = training2.to_dataloader(train=True,  batch_size=BS,   num_workers=4, pin_memory=True)
    vdl2 = validation2.to_dataloader(train=False, batch_size=BS*2, num_workers=4)
    best_cfg = dict(hidden=int(win.hidden), lr=float(win.lr),
                    dropout=float(win.dropout), heads=int(win.heads))
    row = run_trial('v2_tiebreak', best_cfg, max_epochs=40, patience=6,
                    train_ds=training2, tdl=tdl2, vdl=vdl2, raw=VAL_RAW)
    pd.DataFrame([row]).to_csv(REPORTS / 'hpo_v2_tiebreak.csv', index=False)
    print(row)
    print('Compare val_mae_green vs the Stage B winner above — val-based selection, as always.')
else:
    print('Skipped (RUN_V2_TIEBREAK = False)')

In [ ]:
# 10. Bundle artifacts for download (Output panel).
import os
for fn in ('hpo_stage_a.csv', 'hpo_stage_b.csv', 'hpo_v2_tiebreak.csv',
           'tft_breakdown.csv', 'tft_calibration.csv', 'tft_recalibration.csv'):
    p = REPORTS / fn
    if p.exists():
        print('\n===', fn, '===')
        print(pd.read_csv(p).to_string(index=False))
(Path(REPO) / 'models').mkdir(exist_ok=True)
!cd $REPO && zip -qr /kaggle/working/tft_hpo_artifacts.zip models reports/lap_time
print('\nDownload: /kaggle/working/tft_hpo_artifacts.zip')

## After downloading (LOCAL steps)
**If the sweep WON (cell 8 deployed a new model):**
1. Unzip `tft_hpo_artifacts.zip` into the repo root (overwrites `models/tft_lap.*`,
   `models/tft_calibration.json`, `reports/lap_time/*.csv`).
2. `python -m src.models.lap_time.evaluate` — folds the new TFT numbers into `model_comparison.csv`.
3. `pytest -q` — all green.
4. Update model card + MASTER_CONTEXT Section 8 (new hparams replace hidden_size 32 etc.).

**If the sweep LOST (v1 kept):**
1. Unzip ONLY `reports/lap_time/hpo_stage_*.csv` into the repo (do NOT touch `models/`).
2. Add one line to the model card + Section 8: "2-stage sweep (6 screen + 2 converged) did not
   beat v1 val green 1.12 — hand-picked config confirmed near-optimal."

Either way: commit the report CSVs (model binaries stay gitignored).